In [ ]:
# NOTE: you CAN change this cell
# If you want to use your own database, download it here
# Test in google colab
!gdown --fuzzy https://drive.google.com/file/d/1mW2G_X_18OWo0jg2Mp0yQEOUssD744Sz/view?usp=sharing -O db.csv

Downloading...
From: https://drive.google.com/uc?id=1mW2G_X_18OWo0jg2Mp0yQEOUssD744Sz
To: /content/db.csv
100% 369k/369k [00:00<00:00, 67.6MB/s]


In [ ]:
# NOTE: you CAN change this cell
# Add more to your needs
# you must place ALL pip install here

In [ ]:
# NOTE: you CAN change this cell
# import your library here
import re
import pandas as pd
import itertools
from typing import NamedTuple

In [ ]:
# NOTE: you MUST change this cell
# New methods / functions must be written under class Solution.

ALPHABET = 'abcdefghijklmnopqrstuvwxyz0123456789'

def replace_diacritics(input_string: str) -> str:
    # Pre-compiled patterns for better performance
    patterns = [
        (r'[àáạảãâầấậẩẫăằắặẳẵ]', 'a'),
        (r'[èéẹẻẽêềếệểễ]', 'e'),
        (r'[ìíịỉĩ]', 'i'),
        (r'[òóọỏõôồốộổỗơờớợởỡ]', 'o'),
        (r'[ùúụủũưừứựửữ]', 'u'),
        (r'[ỳýỵỷỹ]', 'y'),
        (r'đ', 'd')
    ]

    result = input_string
    for pattern, replacement in patterns:
        result = re.sub(pattern, replacement, result)
    return result

def eliminate_whitespace(input_string: str) -> str:
    # Use string methods first for simple cases, then regex for complex
    if '  ' not in input_string and '\t' not in input_string and '\n' not in input_string:
        return input_string.replace(' ', '')

    # Pre-compiled regex for repeated use
    if not hasattr(eliminate_whitespace, '_regex'):
        eliminate_whitespace._regex = re.compile(r'\s+')
    return eliminate_whitespace._regex.sub('', input_string).strip()

def filter_non_alnum_chars(input_string: str) -> str:
    """Remove characters that are not lowercase alphabets, digits, or spaces."""
    # Pre-compiled regex for better performance
    if not hasattr(filter_non_alnum_chars, '_regex'):
        filter_non_alnum_chars._regex = re.compile(r'[^a-z0-9 ]')
    return filter_non_alnum_chars._regex.sub('', input_string)

# Call before removing accents
def separate_camel_case(input_string: str) -> str:
    """
    Insert space before uppercase letters (specific to Vietnamese names)
    when there are multiple uppercase letters in a word.
    """
    # Pre-compile pattern once
    if not hasattr(separate_camel_case, '_pattern'):
        separate_camel_case._pattern = re.compile(
            r'(?<!^)(?<![\s])([A-ZĐÁÀẢÃẠÂẤẦẨẪẬĂẮẰẲẴẶÉÈẺẼẸÊẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÚÙỦŨỤƯỨỪỬỮỰÝỲỶỸỴ])'
        )

    result = []
    for word in input_string.split():
        # Count uppercase letters more efficiently
        uppercase_count = sum(1 for char in word if char.isupper())
        if uppercase_count > 1:
            # Insert spaces before uppercase letters (except first character)
            processed_word = separate_camel_case._pattern.sub(r' \1', word)
            result.append(processed_word)
        else:
            result.append(word)

    return ' '.join(result)

def standardize_text(input_string: str) -> str:
    # Chain operations more efficiently
    processed = input_string.lower()
    processed = replace_diacritics(processed)
    processed = filter_non_alnum_chars(processed)
    processed = eliminate_whitespace(processed)
    return processed

def preprocess_text(input_string: str) -> str:
    # Split words based on specific patterns
    mapping = {
        r'\bXã': ' Xã ',
        r'\bHuyện': ' Huyện ',
        r'\btỉnh': ' tỉnh ',
        r'\bphố': ' phố ',
        r'\b(T\.T\.H)|(Thừa\.t\.Huế)\b': ' Thừa Thiên Huế ',
        r'\bT\. Hải Dươnwg\b': ' Hải Dương ',
        r'\bFHim\b': 'Hìm',
        r'\bTin GJiang\b': ' Tiền Giang ',
        r'(\d)(?!\d)': r'\1 ',
        r'\b(Thị trấn|Huyện|Thị xã|Thành phố|Tỉnh|khu phố|tp\.|t\.p|tp)\b': ' ',
        r'\bPhường\b': 'P',
        r'\bQuận\b': 'Q',
        r'0(?=[\dA-Za-z])': ''
    }
    text = input_string
    for pat, rep in mapping.items():
        text = re.sub(pat, rep, text, flags=re.IGNORECASE)
    text = re.sub(r'[,-.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(separate_camel_case(w) for w in text.split())


def create_reverse_ngrams(input_string: str, n_values_list=[4, 3, 2, 1]):
    """Generate backward n-grams for various n values."""
    word_list = input_string.split()
    word_count = len(word_list)
    ngram_collection = []

    # Pre-calculate ranges to avoid repeated calculations
    for n_val in n_values_list:
        if word_count >= n_val:
            # Generate indices in reverse order more efficiently
            start_idx = word_count - n_val
            for idx in range(start_idx, -1, -1):
                ngram_text = ' '.join(word_list[idx:idx + n_val])
                ngram_collection.append((idx, ngram_text))

    return ngram_collection

def generate_incorrect_spelling_variants(word: str):
    """
    Generate potential incorrect variants of a word by performing:
      - Replacement: replacing each character with any other character.
      - Deletion: removing a character.
      - Insertion: inserting a new character at each position.
    All operations are adjusted once per character, so maximum edit distance is 1. Need to optimise in worst case.
      - Transposition: swapping adjacent characters. (edit distance 2 but common mistake)
    """
    seen = set()
    length = len(word)
    if 2 < length < 20:
        # Replacement and Deletion
        for i in range(length):
            prefix, char, suffix = word[:i], word[i], word[i+1:]
            # Replacement
            for c in ALPHABET:
                if c != char:
                    variant = prefix + c + suffix
                    if variant not in seen:
                        seen.add(variant)
                        yield variant
            # Deletion
            variant = prefix + suffix
            if variant not in seen:
                seen.add(variant)
                yield variant
        # Insertion
        for i in range(length + 1):
            for c in ALPHABET:
                variant = word[:i] + c + word[i:]
                if variant not in seen:
                    seen.add(variant)
                    yield variant
        # Transposition: swap adjacent characters
        for i in range(length - 1):
            variant_list = list(word)
            variant_list[i], variant_list[i+1] = variant_list[i+1], variant_list[i]
            variant = ''.join(variant_list)
            if variant not in seen:
                seen.add(variant)
                yield variant

def check_valid_sequence(match_objects_list):
    prev_start, prev_end = -1, -1
    for m in match_objects_list:
        if m.start_index == -1:
            continue
        if m.start_index < prev_start or m.start_index < prev_end:
            return False
        prev_start, prev_end = m.start_index, m.end_index
    return True

def get_locations_score(match_objs):
    return sum(m.start_index + len(m.matched_text.split()) * 10 + (1 if m.correct_spelling else 0) for m in match_objs)

class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_terminal = False
        self.raw_name = []

class AddrTrie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str, raw: str):
        """Insert a word and its corresponding raw value into the trie."""
        node = self.root
        for char in word:
            node = node.children.setdefault(char, TrieNode())
        node.is_terminal = True
        if raw not in node.raw_name:
            node.raw_name.append(raw)

    def _get_node(self, word: str):
        """Traverse the trie based on the word and return the node."""
        node = self.root
        for char in word:
            if char not in node.children:
                return None
            node = node.children[char]
        return node

    def search(self, word: str):
        """Search for an exact word in the trie."""
        node = self._get_node(word)
        return (node.is_terminal, node.raw_name) if node and node.is_terminal else (False, None)

    def contain(self, word: str) -> bool:
        """Check if a word exists in the trie."""
        node = self._get_node(word)
        return bool(node and node.is_terminal)

    # Just for debugging
    def traverse(self, node=None, prefix=''):
        """Generator to traverse all words in the trie."""
        node = node or self.root
        if node.is_terminal:
            yield prefix, node.raw_name
        for char, child in node.children.items():
            yield from self.traverse(child, prefix + char)

class MatchResult(NamedTuple):
    start_index: int
    end_index: int
    matched_text: str
    predictions: list
    correct_spelling: bool

EMPTY_MATCH = MatchResult(-1, -1, '', [''], False)

# Main class #
class Solution:
    def __init__(self):
        # list provice, district, ward for private test, do not change for any reason (these file will be provided later with this exact name)

        self.province_path = 'list_province.txt'
        self.district_path = 'list_district.txt'
        self.ward_path = 'list_ward.txt'

        # write your preprocess here, add more method if needed
        self.db_csv = "db.csv"

        self.correct_tries = {}
        self.heuristic_tries = {}
        for level in ["province", "district", "ward"]:
            self.correct_tries[level], self.heuristic_tries[level] = self.build_internal_trie(level)

        self.full_addr_trie = self.build_full_addr_trie()

        # Store external db
        self.use_trie_to_store_external_db = True
        if self.use_trie_to_store_external_db:
            self.ext_province, self.ext_district, self.ext_ward = self.build_external_trie()
        else:
            self.prov = self.store_external_info(self.province_path)
            self.dist = self.store_external_info(self.district_path)
            self.ward = self.store_external_info(self.ward_path)


    def build_internal_trie(self, level):
        correct_trie = AddrTrie()
        incorrect_trie = AddrTrie()

        def insert_edge(raw_name, is_order=False):
            # CHANGED: Use new function names
            norm_val = replace_diacritics(raw_name.lower())
            key = filter_non_alnum_chars(eliminate_whitespace(norm_val))
            remove_prefix = raw_name
            if is_order:
                tokens = raw_name.split()
                if tokens:
                    remove_prefix = str(int(tokens[-1]))
                correct_trie.insert(key, remove_prefix)
            else:
                correct_trie.insert(key, raw_name)

            if not raw_name.isdigit():
                # Abbreviation: first letters of each word
                if not is_order:
                    abbr = ''.join(word[0] for word in norm_val.split())
                    if len(abbr) > 1:
                            correct_trie.insert(abbr, raw_name)

                if is_order:
                    raw_name = remove_prefix

                # Variant: first letters of all words except last + last word
                words = norm_val.split()
                if words:
                    variant2 = ''.join(w[0] for w in words[:-1]) + words[-1]
                    if len(variant2) > 1:
                        correct_trie.insert(variant2, raw_name)

            # Insert possible incorrect variants into heuristic trie
            for variant in generate_incorrect_spelling_variants(key):
                incorrect_trie.insert(variant, raw_name)

        df = pd.read_csv(self.db_csv)
        values = list(set(df[level].str.strip().dropna()))
        for loc in values:
            insert_edge(loc)

        # Add number-type districts and wards
        if level is not None:
            prefix = "Quận" if level == "district" else "Phường"
            max_loc = 12 if level == "district" else 28  # Quận max 12 and Phường max 28
            for order in range(1, max_loc + 1):
                insert_edge(f"{prefix} {order}", True)
                insert_edge(f"{prefix} {order:02d}", True)

        return correct_trie, incorrect_trie

    # This trie only for verfication, not use for detecting
    def build_full_addr_trie(self):
        df = pd.read_csv(self.db_csv)
        combo_trie = AddrTrie()
        for row in df.to_dict(orient='records'):
            city = row['province']
            district = str(int(row['district'])) if str(row['district']).isdigit() else row['district']
            ward = str(int(row['ward'])) if str(row['ward']).isdigit() else row['ward']
            for combo in [ward+district+city, district+city, ward+city, ward+district]:
                # CHANGED: Use new function name
                key = eliminate_whitespace(combo).lower()
                combo_trie.insert(key, key)
        return combo_trie

    # This trie is used for verification, comparing to database user input
    def build_external_trie(self):
        for path in [self.province_path, self.district_path, self.ward_path]:
            trie = AddrTrie()
            with open(path, encoding="utf-8") as file:
                for row in file.readlines():
                    row = row.strip()
                    if row != '':
                        trie.insert(word = row.replace(' ', '').lower(), raw = row)
                        if row.isdigit():
                            trie.insert(word=str(int(row)), raw=row)

            yield trie

    def get_location_matches(self, text, trie, correct_spelling=False):
        return [
            MatchResult(
                start_index=idx,
                end_index=idx + len(ngram.split()),
                matched_text=ngram,
                predictions=raw,
                correct_spelling=correct_spelling
            )
            for idx, ngram in create_reverse_ngrams(text)
            for found, raw in [trie.search(standardize_text(ngram))]
            if found
        ]


    def predict_locations(self, text):
        """Find location matches for province, district, and ward."""
        for location_type in self.locations:
            correct_trie = self.correct_tries[location_type]
            heuristic_trie = self.heuristic_tries[location_type]

            # Get correct spelling matches
            correct_matches = self.get_location_matches(text, correct_trie, correct_spelling=True)
            self.locations[location_type].extend(correct_matches)

            # Get heuristic matches (typo-tolerant)
            heuristic_matches = self.get_location_matches(text, heuristic_trie)
            self.locations[location_type].extend(heuristic_matches)

            # Add empty match option
            self.locations[location_type].append(EMPTY_MATCH)

        return self.locations


    def clear_locations(self):
        self.locations = {"province": [], "district": [], "ward": []}

    def _parse(self, s):
        self.predict_locations(s)
        candidates = []
        max_score = 0

        # Timeout timer
        timer_cnt = 0
        for prov, dist, ward in itertools.product(self.locations['province'],
                                                  self.locations['district'],
                                                  self.locations['ward']):
            # if try 600 times, but still not find best, return the current best
            if timer_cnt > 600:
               break
            # CHANGED: Use new function name
            if not check_valid_sequence([ward, dist, prov]):
                continue
            internal_score = get_locations_score([prov, dist, ward])
            if internal_score < max_score:
                continue

            # # just for debug
            prov_opts = prov.predictions if prov.predictions else ['']
            dist_opts = dist.predictions if dist.predictions else ['']
            ward_opts = ward.predictions if ward.predictions else ['']
            for combo in itertools.product(ward_opts, dist_opts, prov_opts):
                timer_cnt = timer_cnt + 1
                # CHANGED: Use new function name
                key = eliminate_whitespace(''.join(combo)).lower()

                if self.full_addr_trie.contain(key):

                    if internal_score > max_score:
                        max_score = internal_score
                    candidates.append((internal_score, combo))

        if candidates:
            best = max(candidates, key=lambda x: x[0])[1]
        else:
            best = ('', '', '')
            best_score = -1
            for prov, dist, ward in itertools.product(self.locations['province'],
                                                      self.locations['district'],
                                                      self.locations['ward']):
                # CHANGED: Use new function name
                if not check_valid_sequence([ward, dist, prov]):
                    continue
                score = get_locations_score([prov, dist, ward])
                if score > best_score:
                    best_score = score
                    prov_opts = prov.predictions if prov.predictions else ['']
                    dist_opts = dist.predictions if dist.predictions else ['']
                    ward_opts = ward.predictions if ward.predictions else ['']
                    best = (ward_opts[0], dist_opts[0], prov_opts[0])
        self.locations = {
            'province': best[2],
            'district': best[1],
            'ward': best[0]
        }
        return self.locations

    def process(self, s: str):
        # write your process string here
        self.clear_locations()
        s_proc = preprocess_text(s)
        self._parse(s_proc)
        return self.post_process(self.locations)

    @staticmethod
    def store_external_info(fpath):
        res = open(fpath, encoding="utf-8").read().split("\n")
        return [x.strip() for x in res if x.strip() != '']

    def post_process(self, result):
        if self.use_trie_to_store_external_db:
            return {
                    "province": result["province"] if self.ext_province.contain(result["province"].replace(' ', '').lower()) else '',
                    "district": result["district"] if self.ext_district.contain(result["district"].replace(' ', '').lower()) else '',
                    "ward": result["ward"] if self.ext_ward.contain(result["ward"].replace(' ', '').lower()) else '',
                }
        else:
            return {
                "province": result["province"] if result["province"] in self.prov else '',
                "district": result["district"] if result["district"] in self.dist else '',
                "ward": result["ward"] if result["ward"] in self.ward else '',
            }

In [ ]:
# NOTE: DO NOT change this cell
!rm -rf test.json
# this link is public test
!gdown --fuzzy https://drive.google.com/file/d/1PBt3U9I3EH885CDhcXspebyKI5Vw6uLB/view?usp=sharing -O test.json

Downloading...
From: https://drive.google.com/uc?id=1PBt3U9I3EH885CDhcXspebyKI5Vw6uLB
To: /content/test.json
100% 79.4k/79.4k [00:00<00:00, 78.2MB/s]


In [ ]:
# CORRECT TESTS
groups_province = {}
groups_district = {'hòa bình': ['Hoà Bình', 'Hòa Bình'], 'kbang': ['Kbang', 'KBang'], 'quy nhơn': ['Qui Nhơn', 'Quy Nhơn']}
groups_ward = {'ái nghĩa': ['ái Nghĩa', 'Ái Nghĩa'], 'ái quốc': ['ái Quốc', 'Ái Quốc'], 'ái thượng': ['ái Thượng', 'Ái Thượng'], 'ái tử': ['ái Tử', 'Ái Tử'], 'ấm hạ': ['ấm Hạ', 'Ấm Hạ'], 'an ấp': ['An ấp', 'An Ấp'], 'ẳng cang': ['ẳng Cang', 'Ẳng Cang'], 'ẳng nưa': ['ẳng Nưa', 'Ẳng Nưa'], 'ẳng tở': ['ẳng Tở', 'Ẳng Tở'], 'an hòa': ['An Hoà', 'An Hòa'], 'ayun': ['Ayun', 'AYun'], 'bắc ái': ['Bắc ái', 'Bắc Ái'], 'bảo ái': ['Bảo ái', 'Bảo Ái'], 'bình hòa': ['Bình Hoà', 'Bình Hòa'], 'châu ổ': ['Châu ổ', 'Châu Ổ'], 'chư á': ['Chư á', 'Chư Á'], 'chư rcăm': ['Chư Rcăm', 'Chư RCăm'], 'cộng hòa': ['Cộng Hoà', 'Cộng Hòa'], 'cò nòi': ['Cò  Nòi', 'Cò Nòi'], 'đại ân 2': ['Đại Ân  2', 'Đại Ân 2'], 'đak ơ': ['Đak ơ', 'Đak Ơ'], "đạ m'ri": ["Đạ M'ri", "Đạ M'Ri"], 'đông hòa': ['Đông Hoà', 'Đông Hòa'], 'đồng ích': ['Đồng ích', 'Đồng Ích'], 'hải châu i': ['Hải Châu  I', 'Hải Châu I'], 'hải hòa': ['Hải Hoà', 'Hải Hòa'], 'hành tín đông': ['Hành Tín  Đông', 'Hành Tín Đông'], 'hiệp hòa': ['Hiệp Hoà', 'Hiệp Hòa'], 'hòa bắc': ['Hoà Bắc', 'Hòa Bắc'], 'hòa bình': ['Hoà Bình', 'Hòa Bình'], 'hòa châu': ['Hoà Châu', 'Hòa Châu'], 'hòa hải': ['Hoà Hải', 'Hòa Hải'], 'hòa hiệp trung': ['Hoà Hiệp Trung', 'Hòa Hiệp Trung'], 'hòa liên': ['Hoà Liên', 'Hòa Liên'], 'hòa lộc': ['Hoà Lộc', 'Hòa Lộc'], 'hòa lợi': ['Hoà Lợi', 'Hòa Lợi'], 'hòa long': ['Hoà Long', 'Hòa Long'], 'hòa mạc': ['Hoà Mạc', 'Hòa Mạc'], 'hòa minh': ['Hoà Minh', 'Hòa Minh'], 'hòa mỹ': ['Hoà Mỹ', 'Hòa Mỹ'], 'hòa phát': ['Hoà Phát', 'Hòa Phát'], 'hòa phong': ['Hoà Phong', 'Hòa Phong'], 'hòa phú': ['Hoà Phú', 'Hòa Phú'], 'hòa phước': ['Hoà Phước', 'Hòa Phước'], 'hòa sơn': ['Hoà Sơn', 'Hòa Sơn'], 'hòa tân': ['Hoà Tân', 'Hòa Tân'], 'hòa thuận': ['Hoà Thuận', 'Hòa Thuận'], 'hòa tiến': ['Hoà Tiến', 'Hòa Tiến'], 'hòa trạch': ['Hoà Trạch', 'Hòa Trạch'], 'hòa vinh': ['Hoà Vinh', 'Hòa Vinh'], 'hương hòa': ['Hương Hoà', 'Hương Hòa'], 'ích hậu': ['ích Hậu', 'Ích Hậu'], 'ít ong': ['ít Ong', 'Ít Ong'], 'khánh hòa': ['Khánh Hoà', 'Khánh Hòa'], 'krông á': ['Krông Á', 'KRông á'], 'lộc hòa': ['Lộc Hoà', 'Lộc Hòa'], 'minh hòa': ['Minh Hoà', 'Minh Hòa'], 'mường ải': ['Mường ải', 'Mường Ải'], 'mường ẳng': ['Mường ẳng', 'Mường Ẳng'], 'nậm ét': ['Nậm ét', 'Nậm Ét'], 'nam hòa': ['Nam Hoà', 'Nam Hòa'], 'na ư': ['Na ư', 'Na Ư'], 'ngã sáu': ['Ngã sáu', 'Ngã Sáu'], 'nghi hòa': ['Nghi Hoà', 'Nghi Hòa'], 'nguyễn úy': ['Nguyễn Uý', 'Nguyễn úy', 'Nguyễn Úy'], 'nhân hòa': ['Nhân Hoà', 'Nhân Hòa'], 'nhơn hòa': ['Nhơn Hoà', 'Nhơn Hòa'], 'nhơn nghĩa a': ['Nhơn nghĩa A', 'Nhơn Nghĩa A'], 'phúc ứng': ['Phúc ứng', 'Phúc Ứng'], 'phước hòa': ['Phước Hoà', 'Phước Hòa'], 'sơn hóa': ['Sơn Hoá', 'Sơn Hóa'], 'tạ an khương đông': ['Tạ An Khương  Đông', 'Tạ An Khương Đông'], 'tạ an khương nam': ['Tạ An Khương  Nam', 'Tạ An Khương Nam'], 'tăng hòa': ['Tăng Hoà', 'Tăng Hòa'], 'tân hòa': ['Tân Hoà', 'Tân Hòa'], 'tân hòa thành': ['Tân Hòa  Thành', 'Tân Hòa Thành'], 'tân khánh trung': ['Tân  Khánh Trung', 'Tân Khánh Trung'], 'tân lợi': ['Tân lợi', 'Tân Lợi'], 'thái hòa': ['Thái Hoà', 'Thái Hòa'], 'thiết ống': ['Thiết ống', 'Thiết Ống'], 'thuận hòa': ['Thuận Hoà', 'Thuận Hòa'], 'thượng ấm': ['Thượng ấm', 'Thượng Ấm'], 'thụy hương': ['Thuỵ Hương', 'Thụy Hương'], 'thủy xuân': ['Thuỷ Xuân', 'Thủy Xuân'], 'tịnh ấn đông': ['Tịnh ấn Đông', 'Tịnh Ấn Đông'], 'tịnh ấn tây': ['Tịnh ấn Tây', 'Tịnh Ấn Tây'], 'triệu ái': ['Triệu ái', 'Triệu Ái'], 'triệu ẩu': ['Triệu ẩu', 'Triệu Ẩu'], 'trung hòa': ['Trung Hoà', 'Trung Hòa'], 'trung ý': ['Trung ý', 'Trung Ý'], 'tùng ảnh': ['Tùng ảnh', 'Tùng Ảnh'], 'úc kỳ': ['úc Kỳ', 'Úc Kỳ'], 'ứng hòe': ['ứng Hoè', 'Ứng Hoè'], 'vĩnh hòa': ['Vĩnh Hoà', 'Vĩnh Hòa'], 'vũ hòa': ['Vũ Hoà', 'Vũ Hòa'], 'xuân ái': ['Xuân ái', 'Xuân Ái'], 'xuân áng': ['Xuân áng', 'Xuân Áng'], 'xuân hòa': ['Xuân Hoà', 'Xuân Hòa'], 'xuất hóa': ['Xuất Hoá', 'Xuất Hóa'], 'ỷ la': ['ỷ La', 'Ỷ La']}
groups_ward.update({1: ['1', '01'], 2: ['2', '02'], 3: ['3', '03'], 4: ['4', '04'], 5: ['5', '05'], 6: ['6', '06'], 7: ['7', '07'], 8: ['8', '08'], 9: ['9', '09']})
def to_same(groups):
    same = {ele: k for k, v in groups.items() for ele in v}
    return same
same_province = to_same(groups_province)
same_district = to_same(groups_district)
same_ward = to_same(groups_ward)
def normalize(text, same_dict):
    return same_dict.get(text, text)

In [ ]:
TEAM_NAME = 'HK251'
EXCEL_FILE = f'{TEAM_NAME}.xlsx'

import json
import time
with open('test.json', encoding='utf-8') as f:
    data = json.load(f)

summary_only = True
df = []
solution = Solution()
timer = []
correct = 0
for test_idx, data_point in enumerate(data):
    address = data_point["text"]

    ok = 0
    try:
        answer = data_point["result"]
        answer["province_normalized"] = normalize(answer["province"], same_province)
        answer["district_normalized"] = normalize(answer["district"], same_district)
        answer["ward_normalized"] = normalize(answer["ward"], same_ward)

        start = time.perf_counter_ns()
        result = solution.process(address)
        finish = time.perf_counter_ns()
        timer.append(finish - start)
        result["province_normalized"] = normalize(result["province"], same_province)
        result["district_normalized"] = normalize(result["district"], same_district)
        result["ward_normalized"] = normalize(result["ward"], same_ward)

        province_correct = int(answer["province_normalized"] == result["province_normalized"])
        district_correct = int(answer["district_normalized"] == result["district_normalized"])
        ward_correct = int(answer["ward_normalized"] == result["ward_normalized"])
        ok = province_correct + district_correct + ward_correct

        df.append([
            test_idx,
            address,
            answer["province"],
            result["province"],
            answer["province_normalized"],
            result["province_normalized"],
            province_correct,
            answer["district"],
            result["district"],
            answer["district_normalized"],
            result["district_normalized"],
            district_correct,
            answer["ward"],
            result["ward"],
            answer["ward_normalized"],
            result["ward_normalized"],
            ward_correct,
            ok,
            timer[-1] / 1_000_000_000,
        ])
    except Exception as e:
        print(f"{answer = }")
        print(f"{result = }")
        df.append([
            test_idx,
            address,
            answer["province"],
            "EXCEPTION",
            answer["province_normalized"],
            "EXCEPTION",
            0,
            answer["district"],
            "EXCEPTION",
            answer["district_normalized"],
            "EXCEPTION",
            0,
            answer["ward"],
            "EXCEPTION",
            answer["ward_normalized"],
            "EXCEPTION",
            0,
            0,
            0,
        ])
        # any failure count as a zero correct
        pass
    correct += ok


    if not summary_only:
        # responsive stuff
        print(f"Test {test_idx:5d}/{len(data):5d}")
        print(f"Correct: {ok}/3")
        print(f"Time Executed: {timer[-1] / 1_000_000_000:.4f}")


print(f"-"*30)
total = len(data) * 3
score_scale_10 = round(correct / total * 10, 2)
if len(timer) == 0:
    timer = [0]
max_time_sec = round(max(timer) / 1_000_000_000, 4)
avg_time_sec = round((sum(timer) / len(timer)) / 1_000_000_000, 4)

import pandas as pd

df2 = pd.DataFrame(
    [[correct, total, score_scale_10, max_time_sec, avg_time_sec]],
    columns=['correct', 'total', 'score / 10', 'max_time_sec', 'avg_time_sec',],
)

columns = [
    'ID',
    'text',
    'province',
    'province_student',
    'province_normalized',
    'province_student_normalized',
    'province_correct',
    'district',
    'district_student',
    'district_normalized',
    'district_student_normalized',
    'district_correct',
    'ward',
    'ward_student',
    'ward_normalized',
    'ward_student_normalized',
    'ward_correct',
    'total_correct',
    'time_sec',
]

df = pd.DataFrame(df)
df.columns = columns

print(f'{TEAM_NAME = }')
print(f'{EXCEL_FILE = }')
print(df2)

!pip install xlsxwriter
writer = pd.ExcelWriter(EXCEL_FILE, engine='xlsxwriter')
df2.to_excel(writer, index=False, sheet_name='summary')
df.to_excel(writer, index=False, sheet_name='details')
writer.close()

------------------------------
TEAM_NAME = 'HK251'
EXCEL_FILE = 'HK251.xlsx'
   correct  total  score / 10  max_time_sec  avg_time_sec
0     1033   1350        7.65        0.0061         0.002
